# NBA Game-Prediction Dataset Builder

**Source:** [Kaggle - Wyatt Walsh's basketball dataset](https://www.kaggle.com/datasets/wyattowalsh/basketball)
**Input:** `nba.sqlite` (the SQLite DB shipped inside the Kaggle download)

This notebook produces three files:

1. `box_data.csv` — game-level box scores in the T1/T2 layout
2. `team_season_stats.csv` — per team-season aggregates (Elo, Off/Def Eff, Net Rating, last-14-day win pct, seed, etc.)
3. `matchups.csv` — both teams' season features merged onto every game, ready for modeling

T1 is always the home team for that row, T2 is the away team, and `Target_Win = 1` if T1 won.


## Setup

Imports and configuration constants. Edit `DB_PATH` if your SQLite file lives elsewhere.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Config ----
import kagglehub
from kagglehub import KaggleDatasetAdapter
DB_PATH       = "nba.sqlite"          # update if your file is named differently
OUT_DIR       = Path(".")
ELO_BASE      = 1500.0
ELO_K         = 20.0
ELO_HOME_ADV  = 100.0                 # ~3 pts of home-court advantage
ELO_REGRESS   = 0.25                  # 25% regression to the mean each off-season

# Per-game box-score stats we track for both team and opponent
STAT_COLS = ['Score','FGM','FGA','FGM3','FGA3','FTM','FTA',
             'OR','DR','Ast','TO','Stl','Blk','PF']
OPP_COLS  = [f'Opp_{s}' for s in STAT_COLS]

c:\Users\Furag\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Load games and reshape into the T1/T2 box-score format

Pull the home/away rows from the `game` table, then reshape so each row has T1_* (home) and T2_* (away) columns plus `Target_Win`.

In [2]:
def load_games(db_path: str) -> pd.DataFrame:
    """Pull the raw home/away rows from the `game` table."""
    conn = sqlite3.connect(db_path)
    q = """
        SELECT game_id, season_id, game_date,
               team_id_home, team_name_home, wl_home,
               pts_home,  fgm_home,  fga_home,
               fg3m_home, fg3a_home, ftm_home, fta_home,
               oreb_home, dreb_home, ast_home, stl_home,
               blk_home,  tov_home,  pf_home,
               team_id_away, team_name_away,
               pts_away,  fgm_away,  fga_away,
               fg3m_away, fg3a_away, ftm_away, fta_away,
               oreb_away, dreb_away, ast_away, stl_away,
               blk_away,  tov_away,  pf_away
        FROM   game
        WHERE  wl_home IS NOT NULL
          AND  pts_home IS NOT NULL
          AND  pts_away IS NOT NULL
        ORDER  BY game_date
    """
    df = pd.read_sql(q, conn)
    conn.close()
    return df

In [3]:
def create_box_data(games: pd.DataFrame) -> pd.DataFrame:
    """Convert raw home/away rows into the T1/T2 box layout."""
    # season_id like '22023' => regular-season starting in 2023 -> Season = 2023
    games['Season']    = games['season_id'].astype(str).str[-4:].astype(int)
    games['game_date'] = pd.to_datetime(games['game_date'])

    box = pd.DataFrame({
        'Season'      : games['Season'],
        'DayDate'     : games['game_date'],
        'GameID'      : games['game_id'],
        'T1_TeamID'   : games['team_id_home'],
        'T1_TeamName' : games['team_name_home'],
        'T2_TeamID'   : games['team_id_away'],
        'T2_TeamName' : games['team_name_away'],
        'T1_Loc'      : 'H',                     # T1 is always the home team here

        # T1 (home) box
        'T1_Score': games['pts_home'],
        'T1_FGM' : games['fgm_home'],  'T1_FGA' : games['fga_home'],
        'T1_FGM3': games['fg3m_home'], 'T1_FGA3': games['fg3a_home'],
        'T1_FTM' : games['ftm_home'],  'T1_FTA' : games['fta_home'],
        'T1_OR'  : games['oreb_home'], 'T1_DR'  : games['dreb_home'],
        'T1_Ast' : games['ast_home'],  'T1_TO'  : games['tov_home'],
        'T1_Stl' : games['stl_home'],  'T1_Blk' : games['blk_home'],
        'T1_PF'  : games['pf_home'],

        # T2 (away) box
        'T2_Score': games['pts_away'],
        'T2_FGM' : games['fgm_away'],  'T2_FGA' : games['fga_away'],
        'T2_FGM3': games['fg3m_away'], 'T2_FGA3': games['fg3a_away'],
        'T2_FTM' : games['ftm_away'],  'T2_FTA' : games['fta_away'],
        'T2_OR'  : games['oreb_away'], 'T2_DR'  : games['dreb_away'],
        'T2_Ast' : games['ast_away'],  'T2_TO'  : games['tov_away'],
        'T2_Stl' : games['stl_away'],  'T2_Blk' : games['blk_away'],
        'T2_PF'  : games['pf_away'],

        # Target = did T1 win?
        'Target_Win': (games['wl_home'] == 'W').astype(int),
    })

    # Drop rows missing any numeric box stat
    num_cols = [c for c in box.columns
                if c.startswith(('T1_','T2_')) and c not in
                ('T1_TeamID','T1_TeamName','T1_Loc','T2_TeamID','T2_TeamName')]
    box = box.dropna(subset=num_cols).reset_index(drop=True)
    return box

Run Step 1 and peek at the result:

In [4]:
raw = load_games(DB_PATH)
print(f"Loaded {len(raw):,} raw games")

box = create_box_data(raw)
box.to_csv(OUT_DIR / "box_data.csv", index=False)
print(f"Wrote box_data.csv ({len(box):,} rows)")
box.head()

Loaded 65,696 raw games
Wrote box_data.csv (46,649 rows)


,Season,DayDate,GameID,T1_TeamID,T1_TeamName,T2_TeamID,T2_TeamName,T1_Loc,T1_Score,T1_FGM,...,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,Target_Win
0,1979,1980-02-03,0037900001,1610616833,East NBA All Stars East,1610616834,West NBA All Stars West,H,144.0,58.0,...,30.0,37.0,24.0,31.0,34.0,29.0,17.0,16.0,32.0,1
1,1979,1980-02-03,0037900001,1610616833,East NBA All Stars East,1610616834,West NBA All Stars West,H,144.0,58.0,...,30.0,37.0,24.0,31.0,34.0,29.0,17.0,16.0,32.0,1
2,1979,1980-05-04,0047900043,1610612747,Los Angeles Lakers,1610612755,Philadelphia 76ers,H,109.0,48.0,...,22.0,28.0,14.0,26.0,28.0,14.0,12.0,13.0,17.0,1
3,1979,1980-05-07,0047900044,1610612747,Los Angeles Lakers,1610612755,Philadelphia 76ers,H,104.0,48.0,...,21.0,27.0,5.0,29.0,34.0,20.0,14.0,11.0,21.0,0
4,1979,1980-05-10,0047900045,1610612755,Philadelphia 76ers,1610612747,Los Angeles Lakers,H,101.0,45.0,...,23.0,30.0,22.0,34.0,20.0,20.0,5.0,5.0,25.0,0


## Step 2 — Per team-season aggregates

A few helpers, then a single function that produces the season-level feature table.

**What gets computed:**

- Season totals + per-game averages of every box stat (own and opponent)
- Possessions: `FGA - OR + TO + 0.475 * FTA`
- `Avg_Off_Eff = (TotalPts / Poss) * 100`
- `Avg_Def_Eff = (TotalOppPts / OppPoss) * 100`
- `Net_Rating  = Avg_Off_Eff - Avg_Def_Eff`
- `EFG_Pct     = (FGM + 0.5*FGM3) / FGA`
- `TO_Rate     = TO / Poss`
- `OR_Pct      = OR / (OR + Opp_DR)`
- `Win_Pct`, `Home_Win_Pct`, `Away_Win_Pct`, `Last_14_Days_Win_Pct`
- `End_Season_Elo` (chronological walk with home-court bump and off-season regression)
- `seed` (within-season win-pct rank as an NBA-friendly proxy for NCAA seeds)


### Helper: stack the box file so each game contributes two team-perspective rows

In [5]:
def explode_to_team_perspective(box: pd.DataFrame) -> pd.DataFrame:
    """Stack the box file so each game contributes two rows -- one per team."""
    def side(prefix_self, prefix_opp, loc):
        d = pd.DataFrame({
            'Season' : box['Season'],
            'DayDate': box['DayDate'],
            'TeamID' : box[f'{prefix_self}_TeamID'],
            'OppID'  : box[f'{prefix_opp}_TeamID'],
            'Loc'    : loc,
            'Won'    : box['Target_Win'] if prefix_self == 'T1' else 1 - box['Target_Win'],
        })
        for s in STAT_COLS:
            d[s]          = box[f'{prefix_self}_{s}']
            d[f'Opp_{s}'] = box[f'{prefix_opp}_{s}']
        return d

    return pd.concat([side('T1', 'T2', 'H'),
                      side('T2', 'T1', 'A')], ignore_index=True)


def compute_possessions(g: pd.DataFrame) -> pd.DataFrame:
    """Per-game possessions for team and opponent."""
    g = g.copy()
    g['Poss']    = g['FGA']     - g['OR']     + g['TO']     + 0.475 * g['FTA']
    g['OppPoss'] = g['Opp_FGA'] - g['Opp_OR'] + g['Opp_TO'] + 0.475 * g['Opp_FTA']
    return g

### Helper: Elo ratings, walked chronologically with off-season regression

In [6]:
def compute_elo(box: pd.DataFrame,
                base: float = ELO_BASE,
                k: float = ELO_K,
                hca: float = ELO_HOME_ADV,
                regress: float = ELO_REGRESS) -> pd.DataFrame:
    """Walk games chronologically, return end-of-season Elo per (team, season)."""
    box = box.sort_values('DayDate').reset_index(drop=True)
    elo: dict = {}
    season_end: dict = {}
    prev_season = None

    for _, row in box[['Season', 'T1_TeamID', 'T2_TeamID', 'Target_Win']].iterrows():
        season = row['Season']
        if prev_season is not None and season != prev_season:
            # off-season regression toward the mean
            for tid in elo:
                elo[tid] = (1 - regress) * elo[tid] + regress * base
        prev_season = season

        t1, t2 = row['T1_TeamID'], row['T2_TeamID']
        e1, e2 = elo.get(t1, base), elo.get(t2, base)

        # T1 = home; give it the home-court bump for expectations
        exp1   = 1.0 / (1.0 + 10 ** (-((e1 + hca) - e2) / 400.0))
        delta  = k * (row['Target_Win'] - exp1)
        elo[t1] = e1 + delta
        elo[t2] = e2 - delta
        season_end[(t1, season)] = elo[t1]
        season_end[(t2, season)] = elo[t2]

    return pd.DataFrame(
        [{'TeamID': t, 'Season': s, 'End_Season_Elo': v}
         for (t, s), v in season_end.items()]
    )

### Main aggregation: every team-season feature in one table

In [7]:
def compute_team_season_stats(box: pd.DataFrame) -> pd.DataFrame:
    """All per team-season features."""
    tg = explode_to_team_perspective(box)
    tg = compute_possessions(tg)

    # ---- last-14-days window (relative to that team's own season end) ----
    season_end = (tg.groupby(['Season', 'TeamID'])['DayDate']
                    .max().rename('SeasonEnd').reset_index())
    tg = tg.merge(season_end, on=['Season', 'TeamID'])
    tg['_in_last14'] = (tg['SeasonEnd'] - tg['DayDate']).dt.days <= 14

    last14 = (tg[tg['_in_last14']]
              .groupby(['Season', 'TeamID'])['Won']
              .mean().rename('Last_14_Days_Win_Pct').reset_index())

    # ---- home / away win pct ----
    home_wp = (tg[tg['Loc'] == 'H'].groupby(['Season', 'TeamID'])['Won']
                 .mean().rename('Home_Win_Pct').reset_index())
    away_wp = (tg[tg['Loc'] == 'A'].groupby(['Season', 'TeamID'])['Won']
                 .mean().rename('Away_Win_Pct').reset_index())

    # ---- season totals & per-game averages ----
    sum_cols = STAT_COLS + OPP_COLS + ['Poss', 'OppPoss']
    totals = tg.groupby(['Season', 'TeamID'])[sum_cols].sum().reset_index()
    games  = tg.groupby(['Season', 'TeamID']).size().rename('Games').reset_index()
    wins   = tg.groupby(['Season', 'TeamID'])['Won'].sum().rename('Wins').reset_index()

    df = totals.merge(games, on=['Season', 'TeamID']).merge(wins, on=['Season', 'TeamID'])

    # advanced metrics on season totals
    safe_poss     = df['Poss'].replace(0, 1)
    safe_opp_poss = df['OppPoss'].replace(0, 1)
    safe_fga      = df['FGA'].replace(0, 1)
    safe_or_dr    = (df['OR'] + df['Opp_DR']).replace(0, 1)

    df['Avg_Off_Eff'] = (df['Score']     / safe_poss)     * 100      # OffEff
    df['Avg_Def_Eff'] = (df['Opp_Score'] / safe_opp_poss) * 100      # DefEff
    df['Net_Rating']  = df['Avg_Off_Eff'] - df['Avg_Def_Eff']        # NetEff
    df['EFG_Pct']     = (df['FGM'] + 0.5 * df['FGM3']) / safe_fga    # EFGPct
    df['TO_Rate']     = df['TO']  / safe_poss                        # TORate
    df['OR_Pct']      = df['OR']  / safe_or_dr                       # ORPct

    df['Win_Pct']         = df['Wins'] / df['Games']
    df['Overall_Win_Pct'] = df['Win_Pct']     # alias kept so spec matches 1:1

    # per-game averages of every box stat (own + opponent)
    for c in STAT_COLS + OPP_COLS:
        df[f'Avg_{c}'] = df[c] / df['Games']

    # join in the auxiliary tables
    df = (df.merge(home_wp, on=['Season', 'TeamID'], how='left')
            .merge(away_wp, on=['Season', 'TeamID'], how='left')
            .merge(last14,  on=['Season', 'TeamID'], how='left')
            .merge(compute_elo(box), on=['Season', 'TeamID'], how='left'))

    # NBA has no NCAA-style seed -- use within-season win-pct rank as a proxy
    df['seed'] = (df.groupby('Season')['Win_Pct']
                    .rank(method='min', ascending=False).astype(int))

    return df

Run Step 2:

In [8]:
ts = compute_team_season_stats(box)
ts.to_csv(OUT_DIR / "team_season_stats.csv", index=False)
print(f"Wrote team_season_stats.csv ({len(ts):,} team-seasons, {len(ts.columns)} cols)")
ts.head()

Wrote team_season_stats.csv (1,286 team-seasons, 75 cols)


,Season,TeamID,Score,FGM,FGA,FGM3,FGA3,FTM,FTA,OR,...,Avg_Opp_Ast,Avg_Opp_TO,Avg_Opp_Stl,Avg_Opp_Blk,Avg_Opp_PF,Home_Win_Pct,Away_Win_Pct,Last_14_Days_Win_Pct,End_Season_Elo,seed
0,1979,1610612747,657.0,270.0,552.0,0.0,4.0,117.0,144.0,103.0,...,31.000000,15.166667,9.333333,10.000000,22.500000,0.666667,0.666667,0.666667,1519.214494,2
1,1979,1610612755,625.0,258.0,530.0,1.0,16.0,108.0,144.0,57.0,...,26.666667,20.000000,9.166667,6.166667,24.500000,0.333333,0.333333,0.333333,1480.785506,3
2,1979,1610616833,288.0,116.0,248.0,2.0,4.0,54.0,88.0,62.0,...,34.000000,29.000000,17.000000,16.000000,32.000000,1.000000,NaN,1.000000,1514.020123,1
3,1979,1610616834,272.0,106.0,228.0,0.0,8.0,60.0,74.0,48.0,...,34.000000,30.000000,20.000000,9.000000,30.000000,NaN,0.000000,0.000000,1485.979877,4
4,1980,1610612738,579.0,241.0,512.0,3.0,17.0,94.0,129.0,100.0,...,18.000000,13.166667,7.833333,5.500000,20.166667,0.666667,0.666667,0.666667,1519.214494,2


## Step 3 — Build the matchup file (both teams' features side-by-side)

Each game becomes a single row with `T1_*` columns for the home team's season features and `T2_*` columns for the away team's, plus `Target_Win`. Column order matches your spec exactly.

In [9]:
# These are the exact features (without the T1_/T2_ prefix) requested in the spec.
FEATURE_COLS = [
    'Avg_Off_Eff', 'Avg_Def_Eff', 'Net_Rating',
    'Win_Pct', 'End_Season_Elo',
    'Overall_Win_Pct', 'Home_Win_Pct', 'Away_Win_Pct',
    'Avg_Score', 'Avg_FGM', 'Avg_FGA', 'Avg_FGM3', 'Avg_FGA3',
    'Avg_FTM',   'Avg_FTA', 'Avg_OR',  'Avg_DR',
    'Avg_Ast',   'Avg_TO',  'Avg_Stl', 'Avg_Blk', 'Avg_PF',
    'Avg_Opp_Score', 'Avg_Opp_FGM', 'Avg_Opp_FGA',
    'Avg_Opp_FGM3',  'Avg_Opp_FGA3',
    'Avg_Opp_FTM',   'Avg_Opp_FTA',
    'Avg_Opp_OR',    'Avg_Opp_DR',   'Avg_Opp_Ast',
    'Avg_Opp_TO',    'Avg_Opp_Stl',  'Avg_Opp_Blk', 'Avg_Opp_PF',
    'seed', 'Last_14_Days_Win_Pct',
]


def build_matchups(box: pd.DataFrame, team_season: pd.DataFrame) -> pd.DataFrame:
    feats = team_season[['Season', 'TeamID'] + FEATURE_COLS]

    t1 = feats.rename(columns={'TeamID': 'T1_TeamID',
                               **{c: f'T1_{c}' for c in FEATURE_COLS}})
    t2 = feats.rename(columns={'TeamID': 'T2_TeamID',
                               **{c: f'T2_{c}' for c in FEATURE_COLS}})

    m = (box[['Season', 'T1_TeamID', 'T2_TeamID', 'Target_Win']]
            .merge(t1, on=['Season', 'T1_TeamID'])
            .merge(t2, on=['Season', 'T2_TeamID']))

    m['ID'] = (m['Season'].astype(str) + '_'
               + m['T1_TeamID'].astype(str) + '_'
               + m['T2_TeamID'].astype(str))

    ordered = (['T1_TeamID', 'T2_TeamID', 'Season', 'ID']
               + [f'T1_{c}' for c in FEATURE_COLS]
               + [f'T2_{c}' for c in FEATURE_COLS]
               + ['Target_Win'])
    return m[ordered]

Run Step 3:

In [10]:
mm = build_matchups(box, ts)
mm.to_csv(OUT_DIR / "matchups.csv", index=False)
print(f"Wrote matchups.csv ({len(mm):,} matchups, {len(mm.columns)} cols)")
mm.head()

Wrote matchups.csv (46,649 matchups, 81 cols)


,T1_TeamID,T2_TeamID,Season,ID,T1_Avg_Off_Eff,T1_Avg_Def_Eff,T1_Net_Rating,T1_Win_Pct,T1_End_Season_Elo,T1_Overall_Win_Pct,...,T2_Avg_Opp_OR,T2_Avg_Opp_DR,T2_Avg_Opp_Ast,T2_Avg_Opp_TO,T2_Avg_Opp_Stl,T2_Avg_Opp_Blk,T2_Avg_Opp_PF,T2_seed,T2_Last_14_Days_Win_Pct,Target_Win
0,1610616833,1610616834,1979,1979_1610616833_1610616834,100.069493,99.578986,0.490507,1.000000,1514.020123,1.000000,...,31.000000,31.000000,34.000000,30.000000,20.000000,9.000000,30.0,4,0.000000,1
1,1610616833,1610616834,1979,1979_1610616833_1610616834,100.069493,99.578986,0.490507,1.000000,1514.020123,1.000000,...,31.000000,31.000000,34.000000,30.000000,20.000000,9.000000,30.0,4,0.000000,1
2,1610612747,1610612755,1979,1979_1610612747_1610612755,103.074992,98.829855,4.245138,0.666667,1519.214494,0.666667,...,17.166667,34.166667,26.666667,20.000000,9.166667,6.166667,24.5,3,0.333333,1
3,1610612747,1610612755,1979,1979_1610612747_1610612755,103.074992,98.829855,4.245138,0.666667,1519.214494,0.666667,...,17.166667,34.166667,26.666667,20.000000,9.166667,6.166667,24.5,3,0.333333,0
4,1610612755,1610612747,1979,1979_1610612755_1610612747,98.829855,103.074992,-4.245138,0.333333,1480.785506,0.333333,...,9.500000,27.666667,31.000000,15.166667,9.333333,10.000000,22.5,2,0.666667,0


## Done

Three CSVs are now sitting next to this notebook:

- `box_data.csv`
- `team_season_stats.csv`
- `matchups.csv`

`matchups.csv` is the prediction-ready file — feed it to your model with `Target_Win` as the label.